In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import spearmanr
import joblib

In [ ]:
seed=1234

In [7]:
df = pd.read_csv("datasets/final_ds.csv")
target_cols = [f"NCAR{i}" for i in range(1,21)]
subset = target_cols + ['stage', 'date']
df_clean = df.dropna(subset=subset).reset_index(drop=True)

In [8]:
df_clean['date'] = pd.to_datetime(df_clean['date'])

In [9]:
train_mask = (df_clean['date'] >= '2025-01-01') & (df_clean['date'] <= '2025-11-30')
test_mask = (df_clean['date'] >= '2026-01-01') & (df_clean['date'] <= '2026-06-30')

In [10]:
models = {}
eval_metrics = []

for dayX in range(1,21):
    target = f'NCAR{dayX}'
    features = ['polarity',
                'drug_count',
                'Total Revenue', 
                'Operating Revenue', 
                'Cash And Cash Equivalents', 
                'Capital Expenditure', 
                'Stock_Trend_30d', 
                'Stock_Trend_60d',
                'NBI_Trend_30d', 
                'NBI_Trend_60d',
                'isDelisted'
                ]


    X_train = df_clean.loc[train_mask, features]
    y_train = df_clean.loc[train_mask, target]

    X_test = df_clean.loc[test_mask, features]
    y_test = df_clean.loc[test_mask, target]

    # X_train['stage'] = X_train['stage'].astype('category')
    # X_test['stage'] = X_test['stage'].astype('category')

    X_train['isDelisted'] = X_train['isDelisted'].astype('int')
    X_test['isDelisted'] = X_test['isDelisted'].astype('int')

    cont_vars = [c for c in features if (c != 'stage' and c!= 'isDelisted')]
    X_train[cont_vars] = X_train[cont_vars].apply(pd.to_numeric)
    X_test[cont_vars] = X_test[cont_vars].apply(pd.to_numeric)

    y_train = y_train.apply(pd.to_numeric)
    y_test = y_test.apply(pd.to_numeric)

    print(f"Train observation count: {len(X_train)}")
    print(f"Test observation count: {len(X_test)}")

    model = xgb.XGBRegressor(
        objective = 'reg:pseudohubererror',
        huber_slope = 1.0,
        eval_metric = 'mae',
        n_estimators = 100,
        learning_rate = 0.05,
        max_depth=5,
        random_state = 42,
        n_jobs = 1,
        tree_method = "hist",
        enable_categorical = True,
        missing = np.nan
    )


    model.fit(X_train,
            y_train,
            eval_set = [(X_test, y_test)],
            verbose= False
            )

    model.save_model("model.json")

    models[dayX] = model

    y_pred = model.predict(X_test)

    # model evaluation

    rank_ic, p_val = spearmanr(y_test, y_pred)

    eval_metrics.append({
        "Day": dayX,
        "Directional Accuracy": np.mean(np.sign(y_pred) == np.sign(y_test)),
        "Rank IC": rank_ic, 
        "p_val": p_val,
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "MAE": mean_absolute_error(y_test, y_pred),
        "R2": r2_score(y_test, y_pred)
    })

    importance = pd.DataFrame({
        'Features' : features,
        'Importance': models[dayX].feature_importances_
    }).sort_values('Importance', ascending = False)

    print(importance)

    labels = range(1,11)

    deciles = pd.qcut(y_pred, q=10, labels = labels)
    spread = y_test.groupby(deciles).mean()

    print(spread)

Train observation count: 2316
Test observation count: 1477
                     Features  Importance
4   Cash And Cash Equivalents    0.154012
3           Operating Revenue    0.121569
6             Stock_Trend_30d    0.119204
2               Total Revenue    0.102474
5         Capital Expenditure    0.099854
7             Stock_Trend_60d    0.094211
9               NBI_Trend_60d    0.085036
8               NBI_Trend_30d    0.081874
1                  drug_count    0.075579
0                    polarity    0.066188
10                 isDelisted    0.000000
1    -0.047128
2    -0.017141
3     0.052901
4     0.002238
5     0.010174
6     0.062524
7     0.001665
8     0.018214
9    -0.005883
10   -0.002972
Name: NCAR1, dtype: float64
Train observation count: 2316
Test observation count: 1477
                     Features  Importance
4   Cash And Cash Equivalents    0.215800
2               Total Revenue    0.132314
6             Stock_Trend_30d    0.102370
7             Stock_Trend_60d   

In [11]:
eval_metrics = pd.DataFrame(eval_metrics)
print(eval_metrics)

    Day  Directional Accuracy   Rank IC         p_val      RMSE       MAE  \
0     1              0.498307  0.106830  3.889367e-05  0.357972  0.074793   
1     2              0.505755  0.121740  2.703195e-06  1.043282  0.113198   
2     3              0.510494  0.124360  1.635346e-06  0.805226  0.131768   
3     4              0.491537  0.084998  1.076359e-03  0.743209  0.145839   
4     5              0.511848  0.147920  1.119995e-08  0.745431  0.160914   
5     6              0.557211  0.204863  1.849545e-15  0.745042  0.168656   
6     7              0.540961  0.210332  3.132709e-16  0.555147  0.170207   
7     8              0.531483  0.214507  7.809320e-17  0.722848  0.191045   
8     9              0.515911  0.181235  2.259088e-12  1.039821  0.211547   
9    10              0.505078  0.166234  1.292603e-10  0.970203  0.210763   
10   11              0.525389  0.216163  4.464870e-17  1.012146  0.217905   
11   12              0.521327  0.167997  8.183711e-11  0.587215  0.208871   

In [12]:
joblib.dump(models, "xgb_models_1_to_20.joblib")

['xgb_models_1_to_20.joblib']